In [2]:
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord

D_LMC = 49.59 * u.kpc

# R136 position from the Stoop notebook
r136 = SkyCoord(ra=84.6767920*u.deg, dec=-69.1006110*u.deg, frame="icrs")

# Tegkelidis et al. final SN1987A position, ICRS J2016
sn87a = SkyCoord("05h35m27.9884s", "-69d16m11.1134s", frame="icrs")

sep = r136.separation(sn87a)
sep_pc = (sep.radian * D_LMC).to(u.pc)

print("Angular separation:", sep.to(u.deg))
print("Angular separation:", sep.to(u.arcmin))
print("Projected separation:", sep_pc)

for t_myr in [1.0, 1.8, 2.5, 3.0, 5.0, 10.0, 12.0, 14.0]:
    # 1 km/s = 1.022712 pc/Myr
    v_kms = sep_pc.value / t_myr / 1.022712
    mu_masyr = v_kms / (4.74047 * D_LMC.value)
    print(f"{t_myr:4.1f} Myr: {v_kms:7.1f} km/s, {mu_masyr:6.3f} mas/yr")

Angular separation: 0d20m02.04316272s
Angular separation: 20.0341 arcmin
Projected separation: 288.99414070577376 pc
 1.0 Myr:   282.6 km/s,  1.202 mas/yr
 1.8 Myr:   157.0 km/s,  0.668 mas/yr
 2.5 Myr:   113.0 km/s,  0.481 mas/yr
 3.0 Myr:    94.2 km/s,  0.401 mas/yr
 5.0 Myr:    56.5 km/s,  0.240 mas/yr
10.0 Myr:    28.3 km/s,  0.120 mas/yr
12.0 Myr:    23.5 km/s,  0.100 mas/yr
14.0 Myr:    20.2 km/s,  0.086 mas/yr


### Code to check the observed proper motion traceback of SN1987A to R136

In [3]:
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord

D_LMC_kpc = 49.59

r136 = SkyCoord(ra=84.6767920*u.deg, dec=-69.1006110*u.deg, frame="icrs")
sn87a = SkyCoord("05h35m27.9884s", "-69d16m11.1134s", frame="icrs")

# Proper motions in mas/yr
pmra_r136 = 1.654
pmdec_r136 = 0.573

pmra_87a = 1.60
pmdec_87a = 0.44

def tangent_offset_mas(origin, target):
    sep = origin.separation(target).to(u.mas).value
    pa = origin.position_angle(target).rad  # east of north
    x_east = sep * np.sin(pa)
    y_north = sep * np.cos(pa)
    return x_east, y_north

x, y = tangent_offset_mas(r136, sn87a)

mu_x = pmra_87a - pmra_r136
mu_y = pmdec_87a - pmdec_r136

mu2 = mu_x**2 + mu_y**2
t_past_yr = (x*mu_x + y*mu_y) / mu2
d_closest_mas = np.sqrt((x - mu_x*t_past_yr)**2 + (y - mu_y*t_past_yr)**2)

v_rel_kms = 4.74047 * D_LMC_kpc * np.sqrt(mu2)
d_closest_pc = d_closest_mas / 1000 / 206265 * D_LMC_kpc * 1000


print("Relative proper motion:", np.sqrt(mu2), "mas/yr")
print("Relative transverse velocity:", v_rel_kms, "km/s")
print("Closest approach time:", t_past_yr/1e6, "Myr ago")
print("Closest approach distance:", d_closest_pc, "pc")


Relative proper motion: 0.14354441821262146 mas/yr
Relative transverse velocity: 33.74440852685549 km/s
Closest approach time: 6.679956423589484 Myr ago
Closest approach distance: 174.27884894270306 pc
